Analise exploratória respondendo a pergunta 3 do Projeto CaixaVerso, com banco de dados (download kaggle) sobre consultas médicas, analisando a quantidade de faltas e o contexto das mesmas

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df=pd.read_csv('KaggleV2-May-2016.csv')
print (df.head())
df.shape
df.describe
df.info

      PatientId  AppointmentID Gender          ScheduledDay  \
0  2.987250e+13        5642903      F  2016-04-29T18:38:08Z   
1  5.589978e+14        5642503      M  2016-04-29T16:08:27Z   
2  4.262962e+12        5642549      F  2016-04-29T16:19:04Z   
3  8.679512e+11        5642828      F  2016-04-29T17:29:31Z   
4  8.841186e+12        5642494      F  2016-04-29T16:07:23Z   

         AppointmentDay  Age      Neighbourhood  Scholarship  Hipertension  \
0  2016-04-29T00:00:00Z   62    JARDIM DA PENHA            0             1   
1  2016-04-29T00:00:00Z   56    JARDIM DA PENHA            0             0   
2  2016-04-29T00:00:00Z   62      MATA DA PRAIA            0             0   
3  2016-04-29T00:00:00Z    8  PONTAL DE CAMBURI            0             0   
4  2016-04-29T00:00:00Z   56    JARDIM DA PENHA            0             1   

   Diabetes  Alcoholism  Handcap  SMS_received No-show  
0         0           0        0             0      No  
1         0           0        0      

<bound method DataFrame.info of            PatientId  AppointmentID Gender          ScheduledDay  \
0       2.987250e+13        5642903      F  2016-04-29T18:38:08Z   
1       5.589978e+14        5642503      M  2016-04-29T16:08:27Z   
2       4.262962e+12        5642549      F  2016-04-29T16:19:04Z   
3       8.679512e+11        5642828      F  2016-04-29T17:29:31Z   
4       8.841186e+12        5642494      F  2016-04-29T16:07:23Z   
...              ...            ...    ...                   ...   
110522  2.572134e+12        5651768      F  2016-05-03T09:15:35Z   
110523  3.596266e+12        5650093      F  2016-05-03T07:27:33Z   
110524  1.557663e+13        5630692      F  2016-04-27T16:03:52Z   
110525  9.213493e+13        5630323      F  2016-04-27T15:09:23Z   
110526  3.775115e+14        5629448      F  2016-04-27T13:30:56Z   

              AppointmentDay  Age      Neighbourhood  Scholarship  \
0       2016-04-29T00:00:00Z   62    JARDIM DA PENHA            0   
1       2016-

In [32]:
df["No_show_flag"] = np.where(df["No-show"] == "Yes", 1, 0)
df_agrupado = (
    df.groupby(["Scholarship", "Hipertension", "Diabetes"]) ["No_show_flag"]
    .agg(
        taxa_falta_pct=lambda x: round(x.mean() * 100, 2),
        total_pacientes="count",
    )
    .reset_index()
)
mapeamento_sim_nao = {0: "Não", 1:"Sim"}

df_agrupado["Bolsa_Familia"] = df_agrupado["Scholarship"].map(mapeamento_sim_nao
)

df_agrupado["Hipertensão"] = df_agrupado["Hipertension"].map(mapeamento_sim_nao)

df_agrupado["Diabetes"] = df_agrupado["Diabetes"].map(mapeamento_sim_nao)

df_agrupado["Condição_Clinica"] = (
    "HTA: "
    + df_agrupado["Hipertensão"]
    + " | Diab: "
    + df_agrupado["Diabetes"]
)


In [33]:
#Visualização da tabela se tem bolsa familia, situação clinica, taxa de falta e total de pacientes
print ("Tabela Agregada de Taxas de Faltas:")
print(
    df_agrupado[
        [
        "Bolsa_Familia",
        "Condição_Clinica",
        "taxa_falta_pct",
        "total_pacientes",
        ]
    ]   
)

Tabela Agregada de Taxas de Faltas:
  Bolsa_Familia      Condição_Clinica  taxa_falta_pct  total_pacientes
0           Não  HTA: Não | Diab: Não           20.51            78431
1           Não  HTA: Não | Diab: Sim           19.27             1318
2           Não  HTA: Sim | Diab: Não           16.93            13861
3           Não  HTA: Sim | Diab: Sim           17.39             6056
4           Sim  HTA: Não | Diab: Não           24.56             8838
5           Sim  HTA: Não | Diab: Sim           25.18              139
6           Sim  HTA: Sim | Diab: Não           19.53             1454
7           Sim  HTA: Sim | Diab: Sim           20.47              430


In [3]:
#Gráfico 
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

ax = sns.barplot(
    data=df_agrupado,
    x="Condição_Clinica",
    y="taxa_falta_pct",
    hue="Bolsa_Familia",
    palette="Set2"
)
plt.title(
    "Taxa de abstenção médica (%) por Condição Clinica e Situação Socioeconomico",
    fontsize=14,
    pad=15,
    fontweight="bold",
)
plt.xlabel("Diagnóstico Clínico (HTA = Hipertensão | Diab = Diabetes)", fontsize=12)
plt.ylabel("Taxa de Falta (%)", fontsize=12)
plt.legend (title="Beneficiario Bolsa Familia", title_fontsize="11", loc="upper right")
plt.ylim(0, 30)

for p in ax.patches:
    altura = p.get_height()
    if altura > 0:
        ax.annotate (
            f"{altura:.1f}%",
            (p.get_x() + p.get_width()/2.0, altura),
            ha="center",
            va="bottom",
            fontsize=10,
            color="black",
            xytext=(0, 3),
            textcoords="offset points"

        )

plt.tight_layout()
plt.show()        

NameError: name 'sns' is not defined

Gráfico acima demonstra que a taxa de faltas às consultas agendadas são do grupo de beneficiário do Bolsa Familia, independentemente da condição clínica. Concluindo que há a necessidade de uma gestão publica, priorizando horarios em que não afetem os grupos de beneficiarios do Bolsa Familia, impactanto menos no horário de trabalho desse grupo e até mesmo opção de reagendamento via SMS ou via WhatsAPP